# Notebook 1: Data Ingestion & Preprocessing

**Goal**: Load documents from multiple sources, clean them, and inspect the output.

## What we cover:
- Loading PDF, HTML, TXT, and JSON files
- Recursive directory ingestion
- Document metadata inspection
- Text quality analysis


In [ ]:
import sys
sys.path.insert(0, '..')   # add project root to path

import warnings
warnings.filterwarnings('ignore')

from data_pipeline.document_loader import DocumentLoader, Document
import pandas as pd
import matplotlib.pyplot as plt

print('Imports OK')

## 1.1 Create Sample Documents
We'll create a few sample text files to demonstrate ingestion.

In [ ]:
import os
os.makedirs('../sample_data', exist_ok=True)

# Create sample TXT file
with open('../sample_data/company_policy.txt', 'w') as f:
    f.write("""Company Refund Policy

1. ELIGIBILITY
Customers are eligible for a full refund within 30 days of purchase.
After 30 days, only store credit is available.
Digital downloads are non-refundable once accessed.

2. PROCESS
To initiate a refund, contact support@company.com with your order number.
Refunds are processed within 5-7 business days.
Original shipping fees are non-refundable.

3. EXCEPTIONS
Items marked as 'Final Sale' cannot be returned.
Custom orders require a 15% restocking fee.
Damaged items must be reported within 48 hours of delivery.
""")

# Create sample JSON file
import json
products = [
    {"text": "The XR-500 laptop features a 15-inch display, 32GB RAM, and 2TB SSD. Battery life is 12 hours. Price: $1,299.", "id": 1, "category": "laptops"},
    {"text": "The SmartWatch Pro tracks heart rate, sleep, and steps. Water-resistant up to 50m. Price: $299.", "id": 2, "category": "wearables"},
    {"text": "The CoolBreeze AC Unit cools up to 500 sq ft. Energy Star certified. SEER rating of 18. Price: $649.", "id": 3, "category": "appliances"},
]
with open('../sample_data/products.json', 'w') as f:
    json.dump(products, f)

# Create sample Markdown file
with open('../sample_data/faq.md', 'w') as f:
    f.write("""# Frequently Asked Questions

## How do I track my order?
Once your order ships, you will receive a tracking email within 24 hours.
You can also track your order at our website under 'My Orders'.

## What payment methods do you accept?
We accept Visa, Mastercard, American Express, PayPal, and Apple Pay.
All transactions are secured with 256-bit SSL encryption.

## Do you offer international shipping?
Yes, we ship to over 50 countries. International orders typically arrive in 10-15 business days.
Import duties and taxes are the responsibility of the customer.
""")

print('Sample files created in ../sample_data/')

## 1.2 Load Documents

In [ ]:
loader = DocumentLoader()

# Load individual files
txt_docs = loader.load('../sample_data/company_policy.txt')
json_docs = loader.load('../sample_data/products.json', content_key='text', metadata_keys=['id', 'category'])
md_docs   = loader.load('../sample_data/faq.md')

print(f'TXT docs:  {len(txt_docs)}')
print(f'JSON docs: {len(json_docs)}')
print(f'MD docs:   {len(md_docs)}')

In [ ]:
# Load full directory at once
dir_docs = loader.load('../sample_data/', loader_type='directory')
print(f'\nDirectory load: {len(dir_docs)} documents')

for doc in dir_docs:
    print(f"  - {doc.metadata.get('source', 'unknown'):50s}  | {len(doc)} chars | ID: {doc.doc_id}")

## 1.3 Inspect Document Structure

In [ ]:
# Detailed inspection of one document
doc = txt_docs[0]
print('=== Document Inspection ===')
print(f'ID:         {doc.doc_id}')
print(f'Length:     {len(doc)} chars')
print(f'Words:      {doc.word_count}')
print(f'Metadata:   {doc.metadata}')
print(f'\nContent preview:')
print('-' * 40)
print(doc.content[:300])
print('...')

## 1.4 Document Statistics

In [ ]:
all_docs = dir_docs

stats = pd.DataFrame([
    {
        'doc_id': doc.doc_id,
        'source': doc.metadata.get('source', '?').split('/')[-1].split('\\')[-1],
        'file_type': doc.metadata.get('file_type', '?'),
        'chars': len(doc),
        'words': doc.word_count,
    }
    for doc in all_docs
])

print(stats.to_string(index=False))
print(f'\nTotal characters: {stats.chars.sum():,}')
print(f'Total words:      {stats.words.sum():,}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

stats.plot.bar(x='source', y='chars', ax=axes[0], title='Document Lengths (chars)', legend=False)
axes[0].set_xlabel('Document')
axes[0].set_ylabel('Characters')
axes[0].tick_params(axis='x', rotation=30)

stats.plot.bar(x='source', y='words', ax=axes[1], title='Document Lengths (words)', color='orange', legend=False)
axes[1].set_xlabel('Document')
axes[1].set_ylabel('Words')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('doc_stats.png', dpi=100)
plt.show()
print('Chart saved as doc_stats.png')

## 1.5 Load from URL (Live Demo)

In [ ]:
# Uncomment to load from a real URL
# url_docs = loader.load('https://en.wikipedia.org/wiki/Retrieval-augmented_generation')
# print(f'URL doc length: {len(url_docs[0])} chars')
# print(url_docs[0].content[:500])
print('URL loading demo (commented out to avoid external requests)')

## Summary

| Step | Result |
|------|--------|
| Loaded TXT | ✅ 1 doc |
| Loaded JSON | ✅ 3 docs (1 per record) |
| Loaded Markdown | ✅ 1 doc |
| Directory scan | ✅ Auto-detected all formats |

**Next**: Notebook 02 — Embedding Creation